# 10 Repeated rows check

**Question.** About one row in nine is an exact copy of an earlier row. Does that change our conclusions?

**Why it could matter.** The file has no user id, so we cannot tell a repeated user from two different users with the same features. If a copy of a training row lands in the holdout, a model can look better than it is on that row.

**What we do.** We keep every row. Then we score only the holdout rows that have no identical feature twin among the 5M training rows, and compare the models again.

**Inputs:** `data/criteo.parquet`, `results/scores_5m.parquet`

**Outputs:** `results/duplicate_rows_summary.csv`, `results/duplicate_rows_check.csv`

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))
from uplift.data import FEATURES, TRAIN_SIZES, make_splits
from uplift.metrics import qini_coefficient

results_dir = ROOT / "results"
df = pd.read_parquet(ROOT / "data" / "criteo.parquet")
test_ids, pool_ids = make_splits(len(df))
train_ids = pool_ids[: max(TRAIN_SIZES)]
print(f"Rows: {len(df):,}  Holdout: {len(test_ids):,}  Training rows used: {len(train_ids):,}")

Rows: 13,979,592  Holdout: 2,000,000  Training rows used: 5,000,000


## 1. How many rows are repeats

A row is an extra copy if every column, including the outcomes, equals an earlier row.

In [2]:
extra = df.duplicated(keep="first")
summary = pd.DataFrame({
    "group": ["all rows", "extra copies"],
    "rows": [len(df), int(extra.sum())],
    "share_of_rows": [1.0, extra.mean()],
    "visit_rate": [df["visit"].mean(), df.loc[extra, "visit"].mean()],
    "conversion_rate": [df["conversion"].mean(), df.loc[extra, "conversion"].mean()],
    "treated_share": [df["treatment"].mean(), df.loc[extra, "treatment"].mean()],
})
summary.to_csv(results_dir / "duplicate_rows_summary.csv", index=False)
display(summary)

,group,rows,share_of_rows,visit_rate,conversion_rate,treated_share
0,all rows,13979592,1.00000,0.046992,0.002917,0.850000
1,extra copies,1655747,0.11844,0.000020,0.000000,0.958838


## 2. Score only holdout rows with no twin in training

A twin is a holdout row whose 12 features are identical to a row among the 5M training rows.

In [3]:
feature_hash = pd.util.hash_pandas_object(df[FEATURES], index=False).to_numpy()
train_hashes = np.unique(feature_hash[train_ids])

scores = pd.read_parquet(results_dir / "scores_5m.parquet")
assert np.array_equal(np.sort(scores.index.to_numpy()), np.sort(test_ids)), "scores file is not the frozen holdout"
has_twin = np.isin(feature_hash[scores.index.to_numpy()], train_hashes)
no_twin = scores[~has_twin]

rows = []
for model in ["response", "s", "t", "x"]:
    col = f"{model}_visit"
    rows.append({
        "model": model,
        "qini_all_holdout": qini_coefficient(scores["visit"].to_numpy(), scores["treatment"].to_numpy(), scores[col].to_numpy()),
        "qini_no_twin_rows": qini_coefficient(no_twin["visit"].to_numpy(), no_twin["treatment"].to_numpy(), no_twin[col].to_numpy()),
    })
check = pd.DataFrame(rows)
check["gap_vs_response_all"] = check["qini_all_holdout"] - check.loc[0, "qini_all_holdout"]
check["gap_vs_response_no_twin"] = check["qini_no_twin_rows"] - check.loc[0, "qini_no_twin_rows"]
check.to_csv(results_dir / "duplicate_rows_check.csv", index=False)
display(check.round(6))

,model,qini_all_holdout,qini_no_twin_rows,gap_vs_response_all,gap_vs_response_no_twin
0,response,0.003402,0.003699,0.000000,0.000000
1,s,0.003559,0.003904,0.000157,0.000205
2,t,0.003579,0.003940,0.000177,0.000241
3,x,0.003687,0.004068,0.000285,0.000369


In [4]:
twin_share = has_twin.mean()
visit_twin, visit_other = scores.loc[has_twin, "visit"].mean(), scores.loc[~has_twin, "visit"].mean()
same_order = list(check.sort_values("qini_all_holdout")["model"]) == list(check.sort_values("qini_no_twin_rows")["model"])
gaps_grow = bool((check["gap_vs_response_no_twin"].iloc[1:] >= check["gap_vs_response_all"].iloc[1:]).all())

display(Markdown(
    f"{twin_share:.1%} of holdout rows have an identical feature twin in the training rows. "
    f"Their visit rate is {visit_twin:.4f}, against {visit_other:.4f} for the rest. "
    f"Without them the models rank in {'the same' if same_order else 'a different'} order, and the gap to the response baseline is "
    f"{'at least as large' if gaps_grow else 'not always as large'} for every uplift model. "
    + ("The ranking and the gaps do not depend on these rows. " if same_order and gaps_grow
       else "The results change without these rows, so this needs a closer look. ")
    + "The gaps here are point estimates, without bootstrap intervals."
))

12.7% of holdout rows have an identical feature twin in the training rows. Their visit rate is 0.0022, against 0.0535 for the rest. Without them the models rank in the same order, and the gap to the response baseline is at least as large for every uplift model. The ranking and the gaps do not depend on these rows. The gaps here are point estimates, without bootstrap intervals.